# Creating Text Embedding Models

## Creating an Embedding Model

### Load dataset

In [2]:
from datasets import load_dataset

# Load MNLI dataset from GLUE, use subset (50,000)
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

NameError: name 'dataset' is not defined

In [3]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

### Train Model

In [5]:
from sentence_transformers import SentenceTransformer

# Use a base model
embedding_model = SentenceTransformer('bert-base-uncased')

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
from sentence_transformers import losses

# Defien the loss funciton. In softmax loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)

In [7]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for STSB
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [8]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="base_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

In [9]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
100,1.072400
200,0.954800
300,0.889500
400,0.852700
500,0.836800
600,0.849900
700,0.829600
800,0.813100
900,0.803100
1000,0.793200


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.8330583285614228, metrics={'train_runtime': 309.7864, 'train_samples_per_second': 161.402, 'train_steps_per_second': 5.045, 'total_flos': 0.0, 'train_loss': 0.8330583285614228, 'epoch': 1.0})

In [10]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.38350377822741377,
 'spearman_cosine': 0.44731828556745046,
 'pearson_manhattan': 0.435125841795868,
 'spearman_manhattan': 0.4547714512575496,
 'pearson_euclidean': 0.42210155132169214,
 'spearman_euclidean': 0.4490260038278156,
 'pearson_dot': 0.3684871648351224,
 'spearman_dot': 0.3728521950316991,
 'pearson_max': 0.435125841795868,
 'spearman_max': 0.4547714512575496}

### In-Depth Evaluation

In [14]:
from mteb import MTEB

# Choose evaluation task
evaluation = MTEB(tasks=["Banking77Classification"])

# Calculate results
results = evaluation.run(embedding_model)
results

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

[MTEBResults(task_name=Banking77Classification, scores=...)]

In [23]:
from pprint import pprint
pprint(results[0].scores)

{'test': [{'accuracy': 0.47457792207792215,
           'f1': 0.473752004642232,
           'f1_weighted': 0.473752004642232,
           'hf_subset': 'default',
           'languages': ['eng-Latn'],
           'main_score': 0.47457792207792215,
           'scores_per_experiment': [{'accuracy': 0.4694805194805195,
                                      'f1': 0.4685995400768526,
                                      'f1_weighted': 0.4685995400768524},
                                     {'accuracy': 0.46396103896103896,
                                      'f1': 0.4663300072108609,
                                      'f1_weighted': 0.4663300072108609},
                                     {'accuracy': 0.47305194805194806,
                                      'f1': 0.469784591350277,
                                      'f1_weighted': 0.46978459135027695},
                                     {'accuracy': 0.49448051948051946,
                                      'f1': 0.4953042542972

In [24]:
print(results[0].evaluation_time)

27.6010901927948
